# Document Parsing Demo

Human-verified tests for the `DocReader` pipeline. Run each cell and inspect the output directly.

**Prerequisites:** activate the venv and set your API key in `.env` before launching Jupyter.
```bash
source .venv/bin/activate
jupyter notebook demo/demo_doc_parsing.ipynb
```

In [ ]:
import json, sys
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, str(Path('..').resolve()))
load_dotenv('../.env')

from core.llm_adapters import LLMClientFactory
from core.doc_reader import DocReaderFactory
from core.models import FieldSchema

with open('../config/field_schema.json') as f:
    SCHEMAS = [FieldSchema(**s) for s in json.load(f)]

# Change model_id here or switch in config/settings.yaml
LLM = LLMClientFactory.get_client('qwen_local')
print('LLM ready:', type(LLM).__name__)

In [ ]:
import IPython.display as ipd

def parse_and_show(file_path: str):
    """Parse a document and display the result as a formatted table."""
    from IPython.display import display, HTML
    import html

    reader = DocReaderFactory.get_reader(file_path)
    record = reader.read(file_path, SCHEMAS, LLM)

    # Show image/PDF inline if possible
    ext = Path(file_path).suffix.lower()
    if ext in ('.png', '.jpg', '.jpeg'):
        display(ipd.Image(file_path, width=500))
    elif ext == '.pdf':
        display(ipd.HTML(f'<a href="{file_path}" target="_blank">Open PDF: {Path(file_path).name}</a>'))

    # Summary header
    status_color = {'complete': '#2d7a2d', 'parse_failed': '#cc0000', 'unprocessed': '#888'}
    color = status_color.get(record.parse_status, '#888')
    display(HTML(f"""
    <table style='border-collapse:collapse;font-family:monospace;font-size:13px;margin-top:8px'>
      <tr><th style='text-align:left;padding:4px 12px;background:#f0f0f0'>file</th>
          <td style='padding:4px 12px'>{html.escape(Path(file_path).name)}</td></tr>
      <tr><th style='text-align:left;padding:4px 12px;background:#f0f0f0'>doc_type</th>
          <td style='padding:4px 12px'>{html.escape(record.doc_type)}</td></tr>
      <tr><th style='text-align:left;padding:4px 12px;background:#f0f0f0'>parse_status</th>
          <td style='padding:4px 12px;color:{color}'><b>{html.escape(record.parse_status)}</b></td></tr>
      {'<tr><th style="text-align:left;padding:4px 12px;background:#f0f0f0">status_reason</th><td style="padding:4px 12px;color:#888">'+html.escape(record.status_reason)+'</td></tr>' if record.status_reason else ''}
    </table>
    """))

    # Fields table
    if record.fields:
        conf_color = {'high': '#2d7a2d', 'medium': '#b36b00', 'low': '#cc0000'}
        rows = ''.join(
            f"<tr>"
            f"<td style='padding:4px 12px'>{html.escape(f.field_name)}</td>"
            f"<td style='padding:4px 12px'>{html.escape(str(f.origin_value))}</td>"
            f"<td style='padding:4px 12px'><b>{html.escape(str(f.unified_value))}</b></td>"
            f"<td style='padding:4px 12px;color:{conf_color.get(f.confidence,"#888")}'>{f.confidence}</td>"
            f"<td style='padding:4px 12px;color:#888;font-size:11px'>{html.escape((f.validation_note or '')+((' | '+f.confidence_note) if f.confidence_note else ''))}</td>"
            f"</tr>"
            for f in record.fields
        )
        display(HTML(f"""
        <table style='border-collapse:collapse;font-family:monospace;font-size:13px;margin-top:8px;width:100%'>
          <thead><tr style='background:#f0f0f0'>
            <th style='padding:4px 12px;text-align:left'>field</th>
            <th style='padding:4px 12px;text-align:left'>origin_value</th>
            <th style='padding:4px 12px;text-align:left'>unified_value</th>
            <th style='padding:4px 12px;text-align:left'>confidence</th>
            <th style='padding:4px 12px;text-align:left'>notes</th>
          </tr></thead>
          <tbody>{rows}</tbody>
        </table>
        """))
    else:
        display(HTML('<p style="color:#888">No fields extracted.</p>'))

    return record

## CLM-001 — police_report.pdf

In [ ]:
record = parse_and_show('../claims/CLM-001/police_report.pdf')

## CLM-001 — adjuster_note.png

In [ ]:
record = parse_and_show('../claims/CLM-001/adjuster_note.png')

## CLM-001 — finance_agreement.png

In [ ]:
record = parse_and_show('../claims/CLM-001/finance_agreement.png')

## CLM-001 — settlement_breakdown.pdf

In [ ]:
record = parse_and_show('../claims/CLM-001/settlement_breakdown.pdf')

---
## Parse all docs in any claim folder

In [ ]:
CLAIM_DIR = '../claims/CLM-001'   # ← change to any claim folder

supported = {'.pdf', '.png', '.jpg', '.jpeg', '.txt'}
files = sorted(p for p in Path(CLAIM_DIR).iterdir() if p.suffix.lower() in supported)

for f in files:
    print(f'\n{"-"*60}')
    parse_and_show(str(f))